# Features that catch the missed "good" units (fig5e, spontaneous regime)

**goal**: raise the recall of the "good" single-unit classifier substantially while the precision stays within +-1 point of the current model (unpenalized GLM on 10 quality metrics, threshold 0.8).

**what the missed units look like** (analysis of the out-of-fold predictions of the current model): lower firing rate (1.2 vs 1.8 Hz), more variable firing (higher `firing_range`), more variable spike amplitudes (higher `mad_ratio` and `sd_ratio`), lower SNR, fewer spikes (728 vs 1,097), and best agreement with the ground truth closer to the 80% cut-off. Silhouette and the refractory-period metrics do not separate them.

**candidate features** (src/nodes/models/Flc/extra_features.py, none uses the ground truth), grouped by what they describe:
* amplitude distribution (CV, skewness, kurtosis, fraction of small spikes, drift)
* firing pattern (ISI CV, local variation, Fano factor and empty fraction of 5 s bins, trend)
* spatial footprint and depth
* waveform consistency on the main channel
* split/merge candidates (similarity with the most similar other unit, spike coincidence, refractory violations of the union)
* Kilosort cluster properties (`Amplitude`, `ContamPct`)

**validation (nested cross-validation)**: outer = 10 x repeated stratified 4-fold (75% / 25%). For each model, the threshold is chosen on the training units only (5-fold out-of-fold probabilities): the lowest threshold whose out-of-fold precision reaches the target precision (0.92, the current model's precision). So recall is compared at matched precision. The test units are never used to choose anything. Features are rank-gauss transformed (heavy tails).

**controls**: the current model (GLM, threshold 0.8), the same 10 metrics with the same calibrated threshold, and 10 random noise features added to the 10 metrics (what recall gain comes from just having more features).

Note: the candidate groups were chosen after looking at the missed units of the whole dataset, so a small optimistic bias is possible; the noise-feature control helps judge it.

In [1]:
%load_ext autoreload
%autoreload 2
import os, warnings
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
import numpy as np
import pandas as pd
import spikeinterface as si
import spikecurate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn import metrics as skm
warnings.filterwarnings("ignore")

os.chdir("/home/steeve/steeve/epfl/code/spikebias/")
import sys; sys.path.insert(0, os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd()
from src.nodes.models.Flc import dataloader
from src.nodes.models.Flc import extra_features as ef

WE_PATH = "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m_single_units"
SORTING_TRUE_10mn = "dataset/00_raw/ground_truth_npx_spont_10m"
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"
LABEL = "quality_label"
TARGET_PRECISION = 0.92

In [2]:
%%time
We = si.load_waveforms(WE_PATH, with_recording=True)
ids = np.array([int(u) for u in We.unit_ids])
good, bad = dataloader.get_good_and_bad_units(QUALITY_PATH, exp="NS", sorter="KS4")
data = spikecurate.load_dataset(ids, good, bad, We.sorting, si.load_extractor(SORTING_TRUE_10mn), We,
                                load_qm_if_exists=False, delta_time=1.3, job_kwargs=dict(n_jobs=8, chunk_duration="1s", progress_bar=False))
base = list(data["predictors"])
extra = ef.compute_extra_features(We)
dataset = data["dataset"].join(extra).dropna()          # drops the 2 units with a missing silhouette
rng = np.random.default_rng(0)
for k in range(10): dataset[f"noise_{k}"] = rng.normal(size=len(dataset))
y = dataset[LABEL].values.astype(int)
print(dataset.shape, "good:", y.sum(), "bad:", (1 - y).sum())

(182, 49) good: 84 bad: 98
CPU times: user 16.5 s, sys: 5.03 s, total: 21.5 s
Wall time: 1min 46s


## Do the new features separate the units the current model misses? (single-feature ROC area, good units only: detected vs missed)

In [3]:
# detected vs missed good units: out-of-fold detection rate of the current model over 100 random 75/25 splits
from spikecurate.crossval import _split_train_test
rate = {u: [] for u in dataset.index}
for s in range(100):
    tr, te = _split_train_test(dataset, 0.75, s)
    sc = StandardScaler().fit(tr[base]); clf = LogisticRegression(penalty=None, max_iter=5000).fit(sc.transform(tr[base]), tr[LABEL])
    p = clf.predict_proba(sc.transform(te[base]))[:, 1]
    for u, pi in zip(te.index, p): rate[u].append(pi >= 0.8)
dataset["detect_rate"] = [np.mean(rate[u]) for u in dataset.index]
g = dataset[dataset[LABEL] == 1]
det = (g.detect_rate >= 0.5).astype(int)
print(f"good units: {len(g)}; detected {int(det.sum())}, missed {int((1 - det).sum())}")
auc = {f: skm.roc_auc_score(det, g[f]) for f in base + [c for gr in ef.GROUPS.values() for c in gr]}
tab = pd.Series(auc).rename("AUC detected vs missed").to_frame()
tab["|AUC-0.5|"] = (tab.iloc[:, 0] - 0.5).abs()
display(tab.sort_values("|AUC-0.5|", ascending=False).round(2).head(20))

good units: 84; detected 51, missed 33


,AUC detected vs missed,|AUC-0.5|
mad_ratio,0.23,0.27
isi_lv,0.25,0.25
bin_fano,0.25,0.25
isi_cv,0.25,0.25
amp_p10_p90,0.72,0.22
trough_cv,0.28,0.22
amp_cv,0.28,0.22
firing_range,0.29,0.21
amp_lowfrac,0.32,0.18
firing_rate,0.68,0.18


## Nested cross-validation at matched precision

In [4]:
def pipeline(kind):
    if kind == "logit": return make_pipeline(QuantileTransformer(n_quantiles=100, output_distribution="normal"), LogisticRegression(C=0.3, max_iter=5000))
    return make_pipeline(QuantileTransformer(n_quantiles=100, output_distribution="normal"), ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3, random_state=0))

def calibrate(p_oof, y_true, target, min_pred=5):
    """lowest threshold whose out-of-fold precision reaches the target (largest recall)"""
    order = np.argsort(-p_oof); ys = y_true[order]; ps = p_oof[order]
    tp = np.cumsum(ys); n = np.arange(1, len(ys) + 1)
    ok = np.where((tp / n >= target) & (n >= min_pred))[0]
    return ps[ok[-1]] if len(ok) else ps[min(min_pred, len(ps)) - 1]

def recall_at_precision(yt, p, target):
    """largest recall reachable with precision >= target (threshold-free, so it compares models fairly)"""
    pr, rc, _ = skm.precision_recall_curve(yt, p)
    ok = pr >= target
    return rc[ok].max() if ok.any() else 0.0

def prf(yt, p, thr):
    pred = (p >= thr).astype(int)
    return (skm.precision_score(yt, pred, zero_division=0), skm.recall_score(yt, pred, zero_division=0), skm.average_precision_score(yt, p),
            recall_at_precision(yt, p, 0.88), recall_at_precision(yt, p, 0.92))

def one_fold(k, tr, te, X_by_set, y):
    out = {}
    # current model: unpenalized GLM on the 10 metrics, threshold 0.8
    sc = StandardScaler().fit(X_by_set["10 metrics"][tr]); clf = LogisticRegression(penalty=None, max_iter=5000).fit(sc.transform(X_by_set["10 metrics"][tr]), y[tr])
    out[("current model (GLM, thr 0.8)", "-")] = prf(y[te], clf.predict_proba(sc.transform(X_by_set["10 metrics"][te]))[:, 1], 0.8)
    for name, X in X_by_set.items():
        for kind in ("logit", "extra trees"):
            est = pipeline(kind)
            oof = np.zeros(len(tr))
            for a, b in StratifiedKFold(5, shuffle=True, random_state=k).split(X[tr], y[tr]):
                oof[b] = clone(est).fit(X[tr][a], y[tr][a]).predict_proba(X[tr][b])[:, 1]
            thr = calibrate(oof, y[tr], TARGET_PRECISION)
            m = clone(est).fit(X[tr], y[tr])
            out[(name, kind)] = prf(y[te], m.predict_proba(X[te])[:, 1], thr)
    return out

sets = {"10 metrics": base}
for gname, cols in ef.GROUPS.items(): sets["10 metrics + " + gname] = base + cols
sets["10 metrics + all new features"] = base + [c for gr in ef.GROUPS.values() for c in gr]
sets["10 metrics + 10 noise features (control)"] = base + [f"noise_{k}" for k in range(10)]
X_by_set = {n: dataset[c].values for n, c in sets.items()}
folds = list(RepeatedStratifiedKFold(n_splits=4, n_repeats=10, random_state=0).split(np.zeros(len(y)), y))

In [5]:
%%time
res = Parallel(n_jobs=16)(delayed(one_fold)(k, tr, te, X_by_set, y) for k, (tr, te) in enumerate(folds))
R = {key: np.array([r[key] for r in res]) for key in res[0]}
cur = R[("current model (GLM, thr 0.8)", "-")]
ci = lambda x: 1.96 * np.std(x) / np.sqrt(len(x))
rows = []
for (name, kind), a in R.items():
    if kind == "-":
        rows.append({"features": name, "model": kind, "precision": np.median(a[:, 0]), "recall": np.median(a[:, 1]), "AP": np.median(a[:, 2]),
                     "R@P0.88": np.median(a[:, 3]), "R@P0.92": np.median(a[:, 4])}); continue
    b = R[("10 metrics", kind)]          # same model, 10 metrics: the matched baseline (same calibration)
    d = a - b
    rows.append({"features": name, "model": kind, "precision": np.median(a[:, 0]), "recall": np.median(a[:, 1]), "AP": np.median(a[:, 2]),
                 "R@P0.88": np.median(a[:, 3]), "R@P0.92": np.median(a[:, 4]),
                 "dPrec vs 10 metrics": d[:, 0].mean(), "ci": ci(d[:, 0]), "dRec vs 10 metrics": d[:, 1].mean(), "ci ": ci(d[:, 1]),
                 "dR@P0.88": d[:, 3].mean(), "ci  ": ci(d[:, 3]), "dAP": d[:, 2].mean(), "ci   ": ci(d[:, 2])})
T = pd.DataFrame(rows)
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
display(T.round(3).set_index(["features", "model"]))

precision  recall     AP  R@P0.88  R@P0.92  dPrec vs 10 metrics     ci  dRec vs 10 metrics    ci   dR@P0.88   ci      dAP  ci   
features                                 model                                                                                                                                        
current model (GLM, thr 0.8)             -                0.879   0.619  0.895    0.643    0.571                  NaN    NaN                 NaN    NaN       NaN    NaN    NaN    NaN
10 metrics                               logit            0.929   0.571  0.911    0.714    0.619                0.000  0.000               0.000  0.000     0.000  0.000  0.000  0.000
                                         extra trees      0.941   0.619  0.926    0.762    0.690                0.000  0.000               0.000  0.000     0.000  0.000  0.000  0.000
10 metrics + amplitude distribution      logit            0.926   0.429  0.889    0.619    0.571               -0.026  0.052              -0.118  0.032    -0.037  0.046 -0.011  0.006
                                         extra trees      0.969   0.548  0.917    0.714    0.643                0.007  0.018              -0.065  0.039    -0.088  0.053 -0.005  0.008
10 metrics + firing pattern              logit            0.931   0.667  0.923    0.762    0.762               -0.001  0.016               0.086  0.036     0.089  0.046  0.012  0.007
                                         extra trees      0.941   0.667  0.948    0.857    0.762                0.003  0.023               0.033  0.043     0.054  0.051  0.012  0.008
10 metrics + spatial footprint and depth logit            0.913   0.286  0.903    0.786    0.571               -0.058  0.072              -0.235  0.054    -0.004  0.062 -0.015  0.012
                                         extra trees      0.931   0.476  0.918    0.810    0.667                0.007  0.021              -0.132  0.051     0.027  0.046  0.000  0.009
10 metrics + waveform consistency        logit            1.000   0.429  0.902    0.643    0.571                0.001  0.020              -0.090  0.035    -0.037  0.025 -0.008  0.005
                                         extra trees      1.000   0.524  0.916    0.738    0.619                0.006  0.021              -0.090  0.042    -0.082  0.045 -0.010  0.007
10 metrics + split/merge candidates      logit            0.923   0.619  0.935    0.786    0.714               -0.018  0.024               0.033  0.031     0.086  0.045  0.014  0.007
                                         extra trees      0.937   0.667  0.943    0.810    0.762                0.002  0.020               0.039  0.032     0.021  0.041  0.012  0.007
10 metrics + kilosort cluster properties logit            0.967   0.476  0.903    0.667    0.571               -0.016  0.051              -0.079  0.029    -0.023  0.024 -0.006  0.004
                                         extra trees      0.939   0.571  0.918    0.714    0.619               -0.001  0.017              -0.031  0.034    -0.089  0.049 -0.007  0.005
10 metrics + all new features            logit            0.903   0.476  0.915    0.810    0.690               -0.083  0.084              -0.127  0.062     0.035  0.058 -0.001  0.010
                                         extra trees      0.938   0.786  0.957    0.905    0.857                0.000  0.024               0.115  0.053     0.104  0.048  0.032  0.010
10 metrics + 10 noise features (control) logit            0.955   0.381  0.872    0.524    0.381               -0.007  0.016              -0.158  0.041    -0.125  0.042 -0.027  0.007
                                         extra trees      0.972   0.571  0.906    0.714    0.619                0.003  0.022              -0.039  0.044    -0.087  0.049 -0.026  0.006

CPU times: user 174 ms, sys: 323 ms, total: 497 ms
Wall time: 25.2 s
